In [2]:
import polars as pl
import pyarrow.parquet as pq
import sys

filepath = "../Data/processed/sirene_bilan_ML_prets.parquet" 

print("--- Début de la lecture 'bypass' ---")

try:
    print(f"Lecture du fichier via PyArrow : {filepath}")
    table_arrow = pq.read_table(
        filepath,
    )
    
    print("Conversion de la table PyArrow en DataFrame Polars...")
    df_ML = pl.from_arrow(table_arrow)
    
    print("--- SUCCÈS ! ---\n")
    print("Le DataFrame est maintenant dans Polars, prêt pour la transformation.")
    print(df_ML.head())

except Exception as e:
    print(f"\n--- ERREUR ---", file=sys.stderr)
    print(f"Impossible de lire le fichier, même avec PyArrow : {e}", file=sys.stderr)

--- Début de la lecture 'bypass' ---
Lecture du fichier via PyArrow : ../Data/processed/sirene_bilan_ML_prets.parquet
Conversion de la table PyArrow en DataFrame Polars...
--- SUCCÈS ! ---

Le DataFrame est maintenant dans Polars, prêt pour la transformation.
shape: (5, 30)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ siren     ┆ date_clot ┆ CJCK_Tota ┆ EG_Impots ┆ … ┆ delta_Res ┆ delta_CA_ ┆ delta_Res ┆ delta_CA │
│ ---       ┆ ure_exerc ┆ lActifBru ┆ Taxes     ┆   ┆ ultatNet_ ┆ 1an       ┆ ultatNet_ ┆ _2ans    │
│ str       ┆ ice       ┆ t         ┆ ---       ┆   ┆ 1an       ┆ ---       ┆ 2ans      ┆ ---      │
│           ┆ ---       ┆ ---       ┆ i32       ┆   ┆ ---       ┆ i32       ┆ ---       ┆ i32      │
│           ┆ date      ┆ i32       ┆           ┆   ┆ i32       ┆           ┆ i32       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 005420120 ┆ 2016

In [3]:
df_ML.schema

Schema([('siren', String),
        ('date_cloture_exercice', Date),
        ('CJCK_TotalActifBrut', Int32),
        ('EG_ImpotsTaxes', Int32),
        ('FJ_ResultatFinancier', Int32),
        ('FA_ChiffreAffairesVentes', Int32),
        ('HN_RésultatNet', Int32),
        ('DA_TresorerieActive', Int32),
        ('DL_DettesCourtTerme', Int32),
        ('FB_AchatsMarchandises', Int32),
        ('FR_ResultatExceptionnel', Int32),
        ('DF_CapitauxPropres', Int32),
        ('DM_DettesLongTerme', Int32),
        ('AnneeClotureExercice', Int32),
        ('ratio_rentabilite_nette', Float64),
        ('ratio_endettement', Float64),
        ('ratio_marge_brute', Float64),
        ('ratio_capitaux_propres', Float64),
        ('ratio_tresorerie', Float64),
        ('ratio_resultat_financier', Float64),
        ('ratio_resultat_exceptionnel', Float64),
        ('cible_HN_RésultatNet_T_plus_1', Int32),
        ('ResultatNet_T_moins_1', Int32),
        ('CA_T_moins_1', Int32),
        ('ResultatN

In [4]:
import polars as pl
import pyarrow.parquet as pq
import sys

filepath = "../Data/processed/sirene_bilan_EDA.parquet" 

print("--- Début de la lecture 'bypass' ---")

try:
    print(f"Lecture du fichier via PyArrow : {filepath}")
    table_arrow = pq.read_table(
        filepath,
    )
    
    print("Conversion de la table PyArrow en DataFrame Polars...")
    df_bilan_EDA = pl.from_arrow(table_arrow)
    
    print("--- SUCCÈS ! ---\n")
    print("Le DataFrame est maintenant dans Polars, prêt pour la transformation.")
    print(df_bilan_EDA.head())

except Exception as e:
    print(f"\n--- ERREUR ---", file=sys.stderr)
    print(f"Impossible de lire le fichier, même avec PyArrow : {e}", file=sys.stderr)

--- Début de la lecture 'bypass' ---
Lecture du fichier via PyArrow : ../Data/processed/sirene_bilan_EDA.parquet
Conversion de la table PyArrow en DataFrame Polars...
--- SUCCÈS ! ---

Le DataFrame est maintenant dans Polars, prêt pour la transformation.
shape: (5, 21)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ siren     ┆ date_clot ┆ CJCK_Tota ┆ EG_Impots ┆ … ┆ ratio_cap ┆ ratio_tre ┆ ratio_res ┆ ratio_re │
│ ---       ┆ ure_exerc ┆ lActifBru ┆ Taxes     ┆   ┆ itaux_pro ┆ sorerie   ┆ ultat_fin ┆ sultat_e │
│ str       ┆ ice       ┆ t         ┆ ---       ┆   ┆ pres      ┆ ---       ┆ ancier    ┆ xception │
│           ┆ ---       ┆ ---       ┆ i32       ┆   ┆ ---       ┆ f64       ┆ ---       ┆ nel      │
│           ┆ date      ┆ i32       ┆           ┆   ┆ f64       ┆           ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪══════════

In [5]:
# -----------------------------------------------------------------
# Option 1: Jointure Polars (df_bilan et df_bilan_EDA_subset sont des DF Polars)
# -----------------------------------------------------------------
df_bilan_EDA_subset = df_bilan_EDA.select([
    'siren', 
    'date_cloture_exercice', 
    'FR_ResultatExceptionnel'
])

# Utiliser .join() au lieu de .merge(), et NE PAS convertir df_bilan_EDA_subset en Pandas
df_bilan_joined = df_ML.join(
    df_bilan_EDA_subset, 
    on=['siren', 'date_cloture_exercice'], 
    how='left'
)

df_bilan_joined.head(10) # Ceci affichera un Polars DataFrame

# Si vous avez besoin de continuer votre pipeline ML avec Pandas, convertissez ICI :
# df_bilan_joined_pd = df_bilan_joined.to_pandas()

siren,date_cloture_exercice,CJCK_TotalActifBrut,EG_ImpotsTaxes,FJ_ResultatFinancier,FA_ChiffreAffairesVentes,HN_RésultatNet,DA_TresorerieActive,DL_DettesCourtTerme,FB_AchatsMarchandises,FR_ResultatExceptionnel,DF_CapitauxPropres,DM_DettesLongTerme,AnneeClotureExercice,ratio_rentabilite_nette,ratio_endettement,ratio_marge_brute,ratio_capitaux_propres,ratio_tresorerie,ratio_resultat_financier,ratio_resultat_exceptionnel,cible_HN_RésultatNet_T_plus_1,ResultatNet_T_moins_1,CA_T_moins_1,ResultatNet_T_moins_2,CA_T_moins_2,delta_ResultatNet_1an,delta_CA_1an,delta_ResultatNet_2ans,delta_CA_2ans,FR_ResultatExceptionnel_right
str,date,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""005420120""",2016-12-31,31933093,586967,104225,11836,-261053,711840,92013428,0,781843,0,0,2016,-22.055847,2.881444,1.0,0.0,0.022292,8.805762,66.056353,-376691,null,null,null,null,null,null,null,null,781843
"""005420120""",2017-12-31,22684824,441247,98112,26192,-376691,711840,90919571,0,450623,0,0,2017,-14.381911,4.007947,1.0,0.0,0.03138,3.745877,17.204604,-289131,-261053,11836,null,null,-115638,14356,null,null,450623
"""005450119""",2016-12-31,198691,1475,0,0,-1229,14178,-282592,0,0,0,0,2016,-1.2290e9,-1.422269,0.0,0.0,0.071357,0.0,0.0,-1335,null,null,null,null,null,null,null,null,null
"""005450119""",2017-12-31,197400,0,0,0,-1335,14178,-283927,0,0,0,0,2017,-1.3350e9,-1.438333,0.0,0.0,0.071824,0.0,0.0,-1429,-1229,0,null,null,-106,0,null,null,null
"""005520176""",2016-12-31,3220234,0,0,0,52988,1000000,2546782,0,6348948,0,0,2016,5.2988e10,0.790869,0.0,0.0,0.310536,0.0,6.3489e12,120441,null,null,null,null,null,null,null,null,null
"""005520176""",2017-12-31,3462996,1023695,5629599,845235,120441,1000000,2711306,0,6874052,0,0,2017,0.142494,0.782937,1.0,0.0,0.288767,6.660395,8.132711,265484,52988,0,null,null,67453,845235,null,null,null
"""005520242""",2016-12-31,2176500,869141,6287756,10240,135569,2775000,545577,0,7176013,0,0,2016,13.23916,0.250667,1.0,0.0,1.274983,614.038672,700.782519,158504,null,null,null,null,null,null,null,null,null
"""005520242""",2017-12-31,2179093,947593,5930143,10277,158504,2775000,704081,0,6960485,0,0,2017,15.423178,0.323107,1.0,0.0,1.273466,577.030554,677.287633,-147101,135569,10240,null,null,22935,37,null,null,null
"""005580113""",2016-12-31,8714643,0,1618851,0,-190171,1025056,3267633,0,1735749,0,0,2016,-1.9017e11,0.374959,0.0,0.0,0.117625,1.6189e12,1.7357e12,551432,null,null,null,null,null,null,null,null,null
